# PO と光線追跡法の強度比較

PO の `field_map_*.bin` から得た `|E|^2` と、ガウシアン形状のほぼ平面波 ray を用いた幾何光学的な ray density を比較します。

注意: この ray tracing は位相、干渉、回折を含みません。したがって PO の絶対 `|E|^2` と完全一致させるものではなく、主に強度集中位置や分布形状を比較するための基準です。

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
VENV_DIR = (ROOT.parent / ".venv").resolve()
VENV_SITE_PACKAGES = VENV_DIR / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"

if not VENV_SITE_PACKAGES.exists():
    raise RuntimeError(f".venv の site-packages が見つかりません: {VENV_SITE_PACKAGES}")
if str(VENV_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VENV_SITE_PACKAGES))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-comparison-cache")

print(f"current python : {sys.executable}")
print(f"using packages : {VENV_SITE_PACKAGES}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("numpy     ", np.__version__)
print("matplotlib", plt.matplotlib.__version__)

In [ ]:
RESULT_DIRS = [Path("260619_1348"), Path("260621_1559")]
LABELS = [path.name for path in RESULT_DIRS]
OUTDIR = Path("raytrace_comparison")
OUTDIR.mkdir(exist_ok=True)

# PO 側の入射ガウシアンと同じ値。Rayleigh 長が長いので、形状内ではほぼ平面波です。
FREQ_HZ = 94.0e9
C0 = 2.99792458e8
LAMBDA0 = C0 / FREQ_HZ
W0_SRC = 0.2
Z0_SRC = 0.0
ZR_SRC = np.pi * W0_SRC**2 / LAMBDA0

# 軸対称なので、半径方向の annulus ray として追跡します。
N_RAYS = 30000
MAX_REFLECTIONS = 60
AXIS_PROBE_RADIUS = 0.5e-3

print(f"lambda0 = {LAMBDA0:.6e} m")
print(f"zR      = {ZR_SRC:.6e} m")
print(f"output  = {OUTDIR}")

## PO field map の読み込み

In [ ]:
def parse_geometry(path: Path) -> dict[str, float]:
    geometry = {}
    for line in path.read_text().splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.split("[", 1)[0].strip()
        try:
            geometry[key] = float(value.split()[0])
        except ValueError:
            pass
    return geometry


def parse_xy_index(path: Path):
    rows = []
    for line in path.read_text().splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        index, z_value, region = line.split()[:3]
        rows.append((int(index), float(z_value), region))
    return rows


def choose_cone_end_xy_file(result_dir: Path, geometry: dict[str, float]):
    cone_end_z = geometry["base_z"] + geometry["l_cone"]
    rows = parse_xy_index(result_dir / "xy_plane_index_to_z.txt")
    index, z_value, region = min(rows, key=lambda row: abs(row[1] - cone_end_z))
    return result_dir / f"field_map_xy_z{index:03d}.bin", z_value, cone_end_z


def read_field_map(path: Path):
    raw = np.fromfile(path, dtype="<f8")
    if raw.size % 4 != 0:
        raise ValueError(f"{path} は x, y, z, intensity の4平面に分けられません")
    n_points = raw.size // 4
    x, y, z, intensity = np.split(raw, 4)
    reset = np.flatnonzero(np.diff(x) <= 0)
    if reset.size == 0:
        raise ValueError(f"{path}: x 座標からグリッド幅を推定できません")
    nx = int(reset[0] + 1)
    ny = int(n_points // nx)
    shape = (ny, nx)
    return {
        "path": path,
        "nx": nx,
        "ny": ny,
        "x": x.reshape(shape),
        "y": y.reshape(shape),
        "z": z.reshape(shape),
        "intensity": intensity.reshape(shape),
    }


def radial_profile_from_xy(field_map, max_radius, bin_width=None):
    x = field_map["x"]
    y = field_map["y"]
    intensity = field_map["intensity"]
    radius = np.hypot(x, y)
    finite = np.isfinite(intensity)
    if bin_width is None:
        dx = np.diff(x[0, :])
        bin_width = np.min(dx[dx > 0])
    edges = np.arange(0.0, max_radius + bin_width, bin_width)
    index = np.digitize(radius[finite], edges) - 1
    valid = (index >= 0) & (index < len(edges) - 1)
    index = index[valid]
    values = intensity[finite][valid]
    counts = np.bincount(index, minlength=len(edges) - 1)
    sums = np.bincount(index, weights=values, minlength=len(edges) - 1)
    means = np.full(len(edges) - 1, np.nan)
    means[counts > 0] = sums[counts > 0] / counts[counts > 0]
    if np.any(counts > 0):
        last = np.flatnonzero(counts > 0)[-1] + 1
        edges = edges[: last + 1]
        means = means[:last]
        counts = counts[:last]
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, edges, means, counts


def axis_profile_from_xz(field_map):
    x_axis = field_map["x"][0, :]
    z_axis = field_map["z"][:, 0]
    intensity = field_map["intensity"]
    axis_i = np.array([np.interp(0.0, x_axis, row) for row in intensity])
    return z_axis, axis_i


def normalize_profile(values):
    peak = np.nanmax(values)
    if not np.isfinite(peak) or peak == 0.0:
        return values * np.nan
    return values / peak


def best_fit_scale(model, target):
    mask = np.isfinite(model) & np.isfinite(target) & (model > 0)
    if not np.any(mask):
        return np.nan
    return np.sum(model[mask] * target[mask]) / np.sum(model[mask] ** 2)

In [ ]:
po_profiles = []

for result_dir, label in zip(RESULT_DIRS, LABELS):
    geometry = parse_geometry(result_dir / "geometry_summary.txt")
    xy_file, xy_z, cone_end_z = choose_cone_end_xy_file(result_dir, geometry)
    xy_map = read_field_map(xy_file)
    xz_map = read_field_map(result_dir / "field_map_xz.bin")

    r_centers, r_edges, radial_mean, radial_count = radial_profile_from_xy(
        xy_map,
        max_radius=geometry["r_pipe"],
    )
    z_axis, axis_i = axis_profile_from_xz(xz_map)

    po_profiles.append({
        "label": label,
        "result_dir": result_dir,
        "geometry": geometry,
        "xy_file": xy_file,
        "xy_z": xy_z,
        "cone_end_z": cone_end_z,
        "r_centers": r_centers,
        "r_edges": r_edges,
        "radial_mean": radial_mean,
        "radial_count": radial_count,
        "z_axis": z_axis,
        "axis_i": axis_i,
    })

for profile in po_profiles:
    print(f"{profile['label']}: cone end z={profile['cone_end_z']:.6f} m, selected {profile['xy_file']}")

## 軸対称 ray tracing

ここでは `geometry_summary.txt` の半径、長さ、放物面キャップだけを使い、軸対称な円錐+円筒+放物面として追跡します。多角形断面やソフト遷移まで厳密に入れる場合は、壁面交差判定をその形状に置き換えます。

In [ ]:
def gaussian_intensity_on_source_plane(radius, z, w0=W0_SRC, z0=Z0_SRC, zR=ZR_SRC):
    dz = z - z0
    wz = w0 * np.sqrt(1.0 + (dz / zR) ** 2)
    return (w0 / wz) ** 2 * np.exp(-2.0 * radius**2 / wz**2)


def make_trace_geometry(geometry):
    base_z = geometry["base_z"]
    r_cone_in = geometry["r_cone_in"]
    r_pipe = geometry["r_pipe"]
    l_cone = geometry["l_cone"]
    l_pipe = geometry["l_pipe"]
    f_cap = geometry["f_cap"]
    z_cone_end = base_z + l_cone
    z_pipe_end = base_z + l_cone + l_pipe
    z_cap_vertex = z_pipe_end + r_pipe**2 / (4.0 * f_cap)
    return {
        "base_z": base_z,
        "r_cone_in": r_cone_in,
        "r_pipe": r_pipe,
        "l_cone": l_cone,
        "l_pipe": l_pipe,
        "f_cap": f_cap,
        "z_cone_end": z_cone_end,
        "z_pipe_end": z_pipe_end,
        "z_cap_vertex": z_cap_vertex,
        "cone_slope": (r_pipe - r_cone_in) / l_cone,
    }


def wall_radius_axisymmetric(z, geom):
    z = np.asarray(z)
    radius = np.full_like(z, np.nan, dtype=float)
    base_z = geom["base_z"]
    z_cone_end = geom["z_cone_end"]
    z_pipe_end = geom["z_pipe_end"]
    z_cap_vertex = geom["z_cap_vertex"]
    cone = (z >= base_z) & (z <= z_cone_end)
    pipe = (z > z_cone_end) & (z <= z_pipe_end)
    cap = (z > z_pipe_end) & (z <= z_cap_vertex)
    radius[cone] = geom["r_cone_in"] + geom["cone_slope"] * (z[cone] - base_z)
    radius[pipe] = geom["r_pipe"]
    arg = geom["r_pipe"] ** 2 - 4.0 * geom["f_cap"] * (z[cap] - z_pipe_end)
    radius[cap] = np.sqrt(np.maximum(arg, 0.0))
    return radius


def reflect(direction, normal):
    return direction - 2.0 * np.dot(direction, normal) * normal


def next_axisymmetric_hit(point, direction, geom, eps=1.0e-10):
    s0, z0 = point
    us, uz = direction
    base_z = geom["base_z"]
    r_cone_in = geom["r_cone_in"]
    r_pipe = geom["r_pipe"]
    f_cap = geom["f_cap"]
    z_cone_end = geom["z_cone_end"]
    z_pipe_end = geom["z_pipe_end"]
    z_cap_vertex = geom["z_cap_vertex"]
    slope = geom["cone_slope"]
    candidates = []

    # Entrance aperture. Rays are allowed to leave through this plane.
    if uz < -1.0e-14:
        t = (base_z - z0) / uz
        if t > eps:
            s_hit = s0 + t * us
            if abs(s_hit) <= r_cone_in + 1.0e-8:
                candidates.append((t, "exit", None))

    # Cone sides: signed meridional boundaries s = +/- R(z).
    for sign in (1.0, -1.0):
        denom = us - sign * slope * uz
        if abs(denom) > 1.0e-14:
            t = (sign * (r_cone_in + slope * (z0 - base_z)) - s0) / denom
            if t > eps:
                z_hit = z0 + t * uz
                if base_z - 1.0e-9 <= z_hit <= z_cone_end + 1.0e-9:
                    normal = np.array([-sign, slope], dtype=float)
                    normal /= np.linalg.norm(normal)
                    candidates.append((t, "cone", normal))

    # Straight pipe sides.
    if abs(us) > 1.0e-14:
        for sign in (1.0, -1.0):
            t = (sign * r_pipe - s0) / us
            if t > eps:
                z_hit = z0 + t * uz
                if z_cone_end - 1.0e-9 <= z_hit <= z_pipe_end + 1.0e-9:
                    candidates.append((t, "pipe", np.array([-sign, 0.0])))

    # Parabolic cap: z = z_pipe_end + (r_pipe^2 - s^2)/(4 f).
    A = us * us / (4.0 * f_cap)
    B = uz + s0 * us / (2.0 * f_cap)
    C = z0 - z_pipe_end - r_pipe**2 / (4.0 * f_cap) + s0**2 / (4.0 * f_cap)
    roots = []
    if abs(A) < 1.0e-18:
        if abs(B) > 1.0e-18:
            roots.append(-C / B)
    else:
        disc = B * B - 4.0 * A * C
        if disc >= 0.0:
            sqrt_disc = np.sqrt(disc)
            roots.extend([(-B - sqrt_disc) / (2.0 * A), (-B + sqrt_disc) / (2.0 * A)])
    for t in roots:
        if t > eps:
            s_hit = s0 + t * us
            z_hit = z0 + t * uz
            if abs(s_hit) <= r_pipe + 1.0e-8 and z_pipe_end - 1.0e-9 <= z_hit <= z_cap_vertex + 1.0e-9:
                normal = np.array([-s_hit / (2.0 * f_cap), -1.0], dtype=float)
                normal /= np.linalg.norm(normal)
                candidates.append((t, "cap", normal))

    if not candidates:
        return None
    return min(candidates, key=lambda item: item[0])


def trace_axisymmetric_gaussian_rays(geom, n_rays=N_RAYS, max_reflections=MAX_REFLECTIONS):
    r_edges = np.linspace(0.0, geom["r_cone_in"], n_rays + 1)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
    annulus_area = np.pi * (r_edges[1:] ** 2 - r_edges[:-1] ** 2)
    weights = gaussian_intensity_on_source_plane(r_centers, geom["base_z"]) * annulus_area

    segments = []
    hit_counts = {"cone": 0, "pipe": 0, "cap": 0, "exit": 0, "lost": 0}
    shift = 1.0e-9

    for ray_id, (radius, weight) in enumerate(zip(r_centers, weights)):
        point = np.array([radius, geom["base_z"] + shift], dtype=float)
        direction = np.array([0.0, 1.0], dtype=float)

        for order in range(max_reflections + 1):
            hit = next_axisymmetric_hit(point, direction, geom)
            if hit is None:
                hit_counts["lost"] += 1
                break
            distance, kind, normal = hit
            next_point = point + distance * direction
            segments.append([point[0], point[1], next_point[0], next_point[1], weight, ray_id, order])
            hit_counts[kind] += 1
            if kind == "exit":
                break
            direction = reflect(direction, normal)
            direction /= np.linalg.norm(direction)
            point = next_point + shift * direction

    return np.asarray(segments, dtype=float), hit_counts, weights.sum()


def ray_crossings_at_z(segments, z_value):
    z0 = segments[:, 1]
    z1 = segments[:, 3]
    dz = z1 - z0
    z_min = np.minimum(z0, z1)
    z_max = np.maximum(z0, z1)
    mask = (np.abs(dz) > 1.0e-14) & (z_value >= z_min - 1.0e-12) & (z_value < z_max - 1.0e-12)
    t = (z_value - z0[mask]) / dz[mask]
    signed_r = segments[mask, 0] + t * (segments[mask, 2] - segments[mask, 0])
    weights = segments[mask, 4]
    return np.abs(signed_r), weights


def ray_density_on_plane(segments, z_value, radial_edges):
    radius, weights = ray_crossings_at_z(segments, z_value)
    power, _ = np.histogram(radius, bins=radial_edges, weights=weights)
    counts, _ = np.histogram(radius, bins=radial_edges)
    area = np.pi * (radial_edges[1:] ** 2 - radial_edges[:-1] ** 2)
    density = power / area
    return density, counts


def ray_axis_profile(segments, z_axis, probe_radius=AXIS_PROBE_RADIUS):
    area = np.pi * probe_radius**2
    values = np.zeros_like(z_axis, dtype=float)
    counts = np.zeros_like(z_axis, dtype=int)
    for i, z_value in enumerate(z_axis):
        radius, weights = ray_crossings_at_z(segments, z_value)
        inside = radius <= probe_radius
        values[i] = weights[inside].sum() / area
        counts[i] = int(np.count_nonzero(inside))
    return values, counts

In [ ]:
trace_geom = make_trace_geometry(po_profiles[0]["geometry"])
segments, hit_counts, total_ray_power = trace_axisymmetric_gaussian_rays(trace_geom)

radial_edges = po_profiles[0]["r_edges"]
radial_centers = 0.5 * (radial_edges[:-1] + radial_edges[1:])
ray_radial_density, ray_radial_counts = ray_density_on_plane(
    segments,
    po_profiles[0]["cone_end_z"],
    radial_edges,
)

z_axis = po_profiles[0]["z_axis"]
ray_axis_density, ray_axis_counts = ray_axis_profile(segments, z_axis)

print(f"segments        = {len(segments)}")
print(f"total ray power = {total_ray_power:.6e}")
print(f"hit counts      = {hit_counts}")
print(f"radial peak     = {np.nanmax(ray_radial_density):.6e} at r={radial_centers[np.nanargmax(ray_radial_density)]*1e3:.3f} mm")
print(f"axis peak       = {np.nanmax(ray_axis_density):.6e} at z={z_axis[np.nanargmax(ray_axis_density)]*1e3:.3f} mm")

## 規格化比較

ray tracing は干渉を持たないため、まず各曲線を最大値で割って形状を比較します。

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(radial_centers * 1e3, normalize_profile(ray_radial_density), "k--", linewidth=2.2, label="ray trace")
for profile in po_profiles:
    plt.plot(profile["r_centers"] * 1e3, normalize_profile(profile["radial_mean"]), label=f"PO {profile['label']}")
plt.xlabel("r [mm]")
plt.ylabel("normalized intensity")
plt.title(f"Cone-end radial profile, z={po_profiles[0]['cone_end_z']*1e3:.3f} mm")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "radial_normalized_raytrace_vs_po.svg")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(z_axis * 1e3, normalize_profile(ray_axis_density), "k--", linewidth=2.2, label=f"ray trace, r<{AXIS_PROBE_RADIUS*1e3:.1f} mm")
for profile in po_profiles:
    plt.plot(profile["z_axis"] * 1e3, normalize_profile(profile["axis_i"]), label=f"PO {profile['label']}")
plt.xlabel("z [mm]")
plt.ylabel("normalized intensity")
plt.title("Axis profile")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "axis_normalized_raytrace_vs_po.svg")
plt.show()

## 最小二乗スケールで重ねる

参考として、ray density にスケール係数を掛けて PO に最小二乗で合わせた図も作ります。

In [ ]:
fig, axes = plt.subplots(1, len(po_profiles), figsize=(6 * len(po_profiles), 4.6), sharex=True)
if len(po_profiles) == 1:
    axes = [axes]

radial_scales = {}
for ax, profile in zip(axes, po_profiles):
    scale = best_fit_scale(ray_radial_density, profile["radial_mean"])
    radial_scales[profile["label"]] = scale
    ax.plot(profile["r_centers"] * 1e3, profile["radial_mean"], label=f"PO {profile['label']}")
    ax.plot(radial_centers * 1e3, scale * ray_radial_density, "k--", label=f"ray trace x {scale:.3g}")
    ax.set_title(profile["label"])
    ax.set_xlabel("r [mm]")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("intensity")
fig.suptitle("Cone-end radial profile: PO and scaled ray trace")
fig.tight_layout()
fig.savefig(OUTDIR / "radial_scaled_raytrace_vs_po.svg")
plt.show()

radial_scales

In [ ]:
fig, axes = plt.subplots(1, len(po_profiles), figsize=(6 * len(po_profiles), 4.6), sharex=True)
if len(po_profiles) == 1:
    axes = [axes]

axis_scales = {}
for ax, profile in zip(axes, po_profiles):
    scale = best_fit_scale(ray_axis_density, profile["axis_i"])
    axis_scales[profile["label"]] = scale
    ax.plot(profile["z_axis"] * 1e3, profile["axis_i"], label=f"PO {profile['label']}")
    ax.plot(z_axis * 1e3, scale * ray_axis_density, "k--", label=f"ray trace x {scale:.3g}")
    ax.set_title(profile["label"])
    ax.set_xlabel("z [mm]")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("intensity")
fig.suptitle("Axis profile: PO and scaled ray trace")
fig.tight_layout()
fig.savefig(OUTDIR / "axis_scaled_raytrace_vs_po.svg")
plt.show()

axis_scales

## ray path の確認

In [ ]:
plt.figure(figsize=(7, 6))
z_wall = np.linspace(trace_geom["base_z"], trace_geom["z_cap_vertex"], 500)
r_wall = wall_radius_axisymmetric(z_wall, trace_geom)
plt.plot(z_wall * 1e3, r_wall * 1e3, color="0.2", linewidth=2)
plt.plot(z_wall * 1e3, -r_wall * 1e3, color="0.2", linewidth=2)

sample_ids = np.linspace(0, N_RAYS - 1, 90, dtype=int)
for ray_id in sample_ids:
    ray_segments = segments[segments[:, 5] == ray_id]
    for seg in ray_segments:
        plt.plot([seg[1] * 1e3, seg[3] * 1e3], [seg[0] * 1e3, seg[2] * 1e3], color="#1f77b4", alpha=0.16, linewidth=0.7)

plt.xlabel("z [mm]")
plt.ylabel("signed r [mm]")
plt.title("Sample ray paths in meridional plane")
plt.axis("equal")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(OUTDIR / "sample_ray_paths_xz.svg")
plt.show()

## CSV と summary の保存

In [ ]:
radial_columns = [
    radial_centers,
    radial_centers * 1e3,
    ray_radial_density,
    normalize_profile(ray_radial_density),
    ray_radial_counts,
]
radial_header = ["r_m", "r_mm", "raytrace_density", "raytrace_norm", "raytrace_count"]
for profile in po_profiles:
    radial_columns += [profile["radial_mean"], normalize_profile(profile["radial_mean"])]
    radial_header += [f"PO_{profile['label']}_mean", f"PO_{profile['label']}_norm"]
np.savetxt(
    OUTDIR / "radial_raytrace_vs_po.csv",
    np.column_stack(radial_columns),
    delimiter=",",
    header=",".join(radial_header),
    comments="",
)

axis_columns = [
    z_axis,
    z_axis * 1e3,
    ray_axis_density,
    normalize_profile(ray_axis_density),
    ray_axis_counts,
]
axis_header = ["z_m", "z_mm", "raytrace_axis_density", "raytrace_axis_norm", "raytrace_count"]
for profile in po_profiles:
    axis_columns += [profile["axis_i"], normalize_profile(profile["axis_i"])]
    axis_header += [f"PO_{profile['label']}_axis", f"PO_{profile['label']}_norm"]
np.savetxt(
    OUTDIR / "axis_raytrace_vs_po.csv",
    np.column_stack(axis_columns),
    delimiter=",",
    header=",".join(axis_header),
    comments="",
)

summary_lines = [
    "Ray tracing vs PO comparison",
    "",
    f"n_rays = {N_RAYS}",
    f"max_reflections = {MAX_REFLECTIONS}",
    f"axis_probe_radius_m = {AXIS_PROBE_RADIUS:.12e}",
    f"total_ray_power = {total_ray_power:.12e}",
    f"segments = {len(segments)}",
    f"hit_counts = {hit_counts}",
    "",
    f"ray_radial_peak_r_m = {radial_centers[np.nanargmax(ray_radial_density)]:.12e}",
    f"ray_radial_peak_density = {np.nanmax(ray_radial_density):.12e}",
    f"ray_axis_peak_z_m = {z_axis[np.nanargmax(ray_axis_density)]:.12e}",
    f"ray_axis_peak_density = {np.nanmax(ray_axis_density):.12e}",
    "",
]

for profile in po_profiles:
    radial_peak_index = np.nanargmax(profile["radial_mean"])
    axis_peak_index = np.nanargmax(profile["axis_i"])
    summary_lines += [
        f"[{profile['label']}]",
        f"po_radial_peak_r_m = {profile['r_centers'][radial_peak_index]:.12e}",
        f"po_radial_peak_intensity = {profile['radial_mean'][radial_peak_index]:.12e}",
        f"po_axis_peak_z_m = {profile['z_axis'][axis_peak_index]:.12e}",
        f"po_axis_peak_intensity = {profile['axis_i'][axis_peak_index]:.12e}",
        f"radial_best_fit_scale_for_raytrace = {radial_scales[profile['label']]:.12e}",
        f"axis_best_fit_scale_for_raytrace = {axis_scales[profile['label']]:.12e}",
        "",
    ]

(OUTDIR / "summary.txt").write_text("\n".join(summary_lines))
print("\n".join(summary_lines))
print(f"saved to {OUTDIR}")